# 3D Taylor-Green Turbulence - Kinetic Energy Budget

**What this notebook does**

1. Runs a pseudo-spectral Direct Numerical Simulation (DNS) of the Taylor-Green vortex, so the velocity field is **generated here** - no large data file needs to be downloaded or committed to the repository.
2. Feeds the resulting velocity field `u(x, y, z, t)` into [`postprocessing.plot_turbulence_budget`](../postprocessing/turbulence_budget.py), which computes and plots the resolved kinetic-energy budget: **production**, **transport** and **dissipation**, plus a global energy-closure diagnostic.
3. Shows the energy spectrum `E(k)` and the spectral energy flux `Pi(k)` for context.

**Requirements**

`numpy`, `scipy`, `matplotlib` (and Jupyter to run this notebook). See `requirements.txt` and the README's *Environment Setup* section.

**Runtime**

The default `PRESET = "quick"` (`N = 32`) finishes in a few seconds on a laptop CPU. `PRESET = "full"` (`N = 64`) reproduces the case used for the figures in `figures/` and takes roughly 20-40 s, using about 0.5 GB of RAM for the stored snapshots.

> **This notebook used to embed the function definition inline.** It now imports it from the `postprocessing` package so there is only one copy of the code to maintain.


## 0. Setup

The bootstrap below makes the repository root importable so that `from postprocessing import ...` works whether you launch Jupyter from the repo root or from a subdirectory, and also on Google Colab after cloning.


In [ ]:
import os
import sys
import time

# Make the postprocessing package importable from anywhere inside the repo.
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "postprocessing")) and REPO_ROOT != os.path.dirname(REPO_ROOT):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import scipy
import scipy.fft as sfft
import matplotlib
import matplotlib.pyplot as plt

from postprocessing import plot_turbulence_budget

print("numpy     :", np.__version__)
print("scipy     :", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("module    :", plot_turbulence_budget.__module__)


## 1. Taylor-Green vortex DNS

Initial condition (a triply periodic domain of size `2*pi` in each direction):

```
u =  sin(x) cos(y) cos(z)
v = -cos(x) sin(y) cos(z)
w =  0
```

The exact initial kinetic energy per unit mass is `KE_0 = 0.125`, which the run below reproduces.

Pick a preset. `velocity_history` comes out with shape `(nt, 3, nx, ny, nz)` - exactly the layout `plot_turbulence_budget` expects for `component_axis=1`.

| Preset | N | dt | t_end | snap_every | snapshots | RAM for snapshots |
|---|---|---|---|---|---|---|
| `quick` | 32 | 0.02 | 20 | 20 | ~51 | ~20 MB |
| `full` | 64 | 0.02 | 30 | 10 | 151 | ~475 MB |


In [ ]:
# ============================================================
# 3D TAYLOR-GREEN VORTEX DNS  (pseudo-spectral, Heun/RK2)
#
# Produces:
#   velocity_history  -> (nt, 3, nx, ny, nz), float32, physical space
#   snapshot_times    -> (nt,) in seconds
#   spectrum_snaps    -> E(k) and Pi(k) per snapshot, for section 3
#
# Requirements: numpy, scipy
# ============================================================

PRESET = "quick"      # "quick" (N=32, seconds) or "full" (N=64, ~20-40 s)

PRESETS = {
    "quick": dict(N=32, t_end=20.0, snap_every=20),
    "full":  dict(N=64, t_end=30.0, snap_every=10),
}

N          = PRESETS[PRESET]["N"]
t_end      = PRESETS[PRESET]["t_end"]
snap_every = PRESETS[PRESET]["snap_every"]

nu = 1 / 400          # kinematic viscosity (Re = 1/nu); N=64 resolves up to about Re 400-600
dt = 0.02             # time step (CFL ~ 0.2 at N=64)

SAVE_NPZ      = False  # True writes taylor_green_dns.npz (git-ignored; see section 4)
SAVE_VORTICITY = False # True stores a vorticity slice per snapshot for the animation in section 3

# Physical domain (periodic cube)
Lx = Ly = Lz = 2 * np.pi
domain_lengths = (Lx, Ly, Lz)

# ------------------------------------------------------------
# Spatial grid
# ------------------------------------------------------------
x = np.arange(N) * Lx / N
y = np.arange(N) * Ly / N
z = np.arange(N) * Lz / N
X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

# ------------------------------------------------------------
# Wavenumbers (rfftn layout: the last axis is a half-spectrum)
# ------------------------------------------------------------
kx = 2 * np.pi * np.fft.fftfreq(N, d=Lx / N)
ky = 2 * np.pi * np.fft.fftfreq(N, d=Ly / N)
kz = 2 * np.pi * np.fft.rfftfreq(N, d=Lz / N)

KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing="ij")
Kvec = np.array([KX, KY, KZ])
K2 = KX**2 + KY**2 + KZ**2

K2inv = np.zeros_like(K2)
K2inv[K2 > 0] = 1.0 / K2[K2 > 0]

# ------------------------------------------------------------
# 2/3-rule dealiasing mask
# ------------------------------------------------------------
kcut = N // 3
dealias_mask = (
    (np.abs(KX) <= kcut) &
    (np.abs(KY) <= kcut) &
    (np.abs(KZ) <= kcut)
)

# ------------------------------------------------------------
# Fourier operators
# ------------------------------------------------------------
def fft3(f):
    return sfft.rfftn(f, axes=(-3, -2, -1), workers=-1)


def ifft3(fh):
    return sfft.irfftn(fh, s=(N, N, N), axes=(-3, -2, -1), workers=-1)


def curl_hat(uh):
    return 1j * np.array([
        KY * uh[2] - KZ * uh[1],
        KZ * uh[0] - KX * uh[2],
        KX * uh[1] - KY * uh[0],
    ])


def nonlinear(uh):
    """Dealiased (u x omega) term, projected to be divergence-free."""
    u = ifft3(uh)
    omega = ifft3(curl_hat(uh))

    cross = np.array([
        u[1] * omega[2] - u[2] * omega[1],
        u[2] * omega[0] - u[0] * omega[2],
        u[0] * omega[1] - u[1] * omega[0],
    ])

    nh = fft3(cross) * dealias_mask
    nh -= Kvec * (np.sum(Kvec * nh, axis=0) * K2inv)   # pressure projection
    return nh


def rhs(uh):
    return nonlinear(uh) - nu * K2 * uh


# ------------------------------------------------------------
# Spectra helpers (used in section 3)
# ------------------------------------------------------------
shell = np.rint(np.sqrt(K2)).astype(int)
wt = np.full(K2.shape, 2.0)     # weight 2 for the rfft modes whose conjugate is omitted
wt[..., 0] = 1.0
wt[..., -1] = 1.0               # Nyquist plane
kk = np.arange(kcut + 1)


def spectra(uh):
    """Shell-summed E(k) and energy flux Pi(k) (positive = forward cascade)."""
    s = 1.0 / N**6
    e_modes = 0.5 * wt * np.sum(np.abs(uh) ** 2, axis=0) * s
    nh = nonlinear(uh)
    t_modes = wt * np.real(np.sum(np.conj(uh) * nh, axis=0)) * s
    E = np.bincount(shell.ravel(), e_modes.ravel(), minlength=N)[: kcut + 1]
    T = np.bincount(shell.ravel(), t_modes.ravel(), minlength=N)[: kcut + 1]
    return E, -np.cumsum(T)


# ------------------------------------------------------------
# Initial condition
# ------------------------------------------------------------
u0 = np.array([
    np.sin(X) * np.cos(Y) * np.cos(Z),
    -np.cos(X) * np.sin(Y) * np.cos(Z),
    np.zeros_like(X),
])

uh = fft3(u0) * dealias_mask

# ------------------------------------------------------------
# Time integration and snapshot storage
# ------------------------------------------------------------
nsteps = int(round(t_end / dt))
nsnapshots = nsteps // snap_every + 1

velocity_history = np.empty((nsnapshots, 3, N, N, N), dtype=np.float32)
snapshot_times = np.empty(nsnapshots)
spectrum_snaps = []

snapshot_index = 0
t_start = time.time()

for n in range(nsteps + 1):

    if n % snap_every == 0:
        velocity_history[snapshot_index] = ifft3(uh)
        snapshot_times[snapshot_index] = n * dt

        E, Pi = spectra(uh)
        snap = {"t": n * dt, "E": E, "Pi": Pi}
        if SAVE_VORTICITY:
            snap["w"] = np.linalg.norm(ifft3(curl_hat(uh))[..., N // 4], axis=0)
        spectrum_snaps.append(snap)

        snapshot_index += 1

    if n < nsteps:
        # Heun (RK2) step
        k1_ = rhs(uh)
        k2_ = rhs(uh + dt * k1_)
        uh = uh + 0.5 * dt * (k1_ + k2_)

elapsed = time.time() - t_start

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("=" * 60)
print(f"TAYLOR-GREEN DNS COMPLETE  (preset: {PRESET})")
print("=" * 60)
print(f"Grid:                  {N} x {N} x {N}")
print(f"Reynolds number:       {1 / nu:.1f}")
print(f"Time step:             {dt}")
print(f"Simulation duration:   {t_end}")
print(f"Snapshot interval:     {snap_every * dt}")
print(f"Number of snapshots:   {snapshot_index}")
print(f"Velocity array shape:  {velocity_history.shape}  ({velocity_history.dtype})")
print(f"Memory (float32):      {velocity_history.nbytes / 1e6:.0f} MB"
      f"   (-> ~{velocity_history.nbytes * 2 / 1e6:.0f} MB as float64)")
print(f"Runtime:               {elapsed:.2f} s")
print("-" * 60)
print(f"Initial KE:            {np.mean(0.5 * np.sum(velocity_history[0] ** 2, axis=0)):.6f}  (exact: 0.1250)")

if SAVE_NPZ:
    np.savez_compressed(
        "taylor_green_dns.npz",
        velocity=velocity_history,
        times=snapshot_times,
        nu=nu,
        dt=dt,
        domain_lengths=domain_lengths,
    )
    print("Saved: taylor_green_dns.npz (git-ignored)")
print("=" * 60)


## 2. Kinetic energy budget

One call computes the budget at every snapshot and draws the figures. `velocity_history` is passed straight from the simulation - nothing has to be loaded from disk.

```python
results, fig, axes = plot_turbulence_budget(
    velocity=velocity_history,
    nu=nu,
    times=snapshot_times,
    domain_lengths=domain_lengths,
    component_axis=1,
    dealias=True,
    show=True,
)
```


In [ ]:
# ============================================================
# KINETIC ENERGY BUDGET
# ============================================================

# Set to e.g. "figures/demo_" to write budget.png, closure.png and terms.png.
SAVE_FIG_PREFIX = None

results, fig, axes = plot_turbulence_budget(
    velocity=velocity_history,
    nu=nu,
    times=snapshot_times,
    domain_lengths=domain_lengths,
    component_axis=1,
    dealias=True,
    show=True,
    save_prefix=SAVE_FIG_PREFIX,
)

print("Velocity array shape:", velocity_history.shape)
print("Snapshots analysed  :", len(snapshot_times))
print("Time range          : %.2f - %.2f s" % (snapshot_times[0], snapshot_times[-1]))
print("Figures created     :", sorted(results["figures"].keys()))


### 2a. The `results` dictionary

| Key | Meaning | Units |
|---|---|---|
| `times` | snapshot times | s |
| `KE` | domain-mean resolved kinetic energy `K = 0.5*(u^2+v^2+w^2)` | m^2/s^2 |
| `production` | production by mean shear - **identically zero here**, see the notes in section 5 | m^2/s^3 |
| `transport_mean` | signed domain mean of `T = -div[(K+p)u] + nu*laplacian(K)` | m^2/s^3 |
| `transport_rms` | RMS of the same field (the meaningful transport magnitude) | m^2/s^3 |
| `dissipation` | `epsilon = 2*nu*S_ij*S_ij` | m^2/s^3 |
| `cumulative_dissipation` | time integral of `epsilon` (trapezoidal) | m^2/s^2 |
| `energy_budget_residual` | `KE + cumulative_dissipation - KE(0)`, i.e. the closure error | m^2/s^2 |
| `figures` | the three matplotlib figures (`budget`, `closure`, `terms`) | - |


In [ ]:
# Numerical summary of the budget
print("Budget summary")
print("-" * 60)
print(f"Initial kinetic energy        : {results['KE'][0]:.6f} m^2/s^2")
print(f"Final kinetic energy          : {results['KE'][-1]:.6f} m^2/s^2")
print(f"Peak dissipation              : {results['dissipation'].max():.6e} m^2/s^3")
print(f"  at t = {results['times'][int(np.argmax(results['dissipation']))]:.2f} s")
print(f"Peak transport RMS            : {results['transport_rms'].max():.6e} m^2/s^3")
print(f"Mean signed transport         : {abs(results['transport_mean']).mean():.3e}  (should be ~0 on a periodic domain)")
print(f"Max |energy closure error|    : {np.abs(results['energy_budget_residual']).max():.3e} m^2/s^2")
print(f"  relative to initial KE      : {np.abs(results['energy_budget_residual']).max() / results['KE'][0]:.3e}")

# You can also pull the arrays out and plot them yourself, without show=True:
# eps = results["dissipation"]; t = results["times"]


## 3. Energy spectrum `E(k)` and flux `Pi(k)`

The budget says *how much* energy is lost to viscosity; the spectra say *at which scale* it happens and in which direction it cascades. `Pi(k) > 0` means a forward cascade from large to small scales.

> The interactive `to_jshtml()` animation is **off by default**: a single embedded animation added ~23 MB of base64 to this notebook file, which makes the repository heavy for every collaborator and unreviewable in a pull request. Set `SHOW_ANIMATION = True` only in your own local copy, and run `Clear All Outputs` before saving.


In [ ]:
# Static spectra: initial, intermediate and final snapshots
SHOW_ANIMATION = False   # keep False in the committed notebook (see note above)

n_show = len(spectrum_snaps)
idx = sorted({0, n_show // 2, n_show - 1})

fig_s, (axE, axP) = plt.subplots(1, 2, figsize=(13, 4.8), layout="constrained")

for j in idx:
    s = spectrum_snaps[j]
    axE.loglog(kk[1:], s["E"][1:], lw=2, label=f"t = {s['t']:.1f} s")
    axP.plot(kk, s["Pi"], lw=2, label=f"t = {s['t']:.1f} s")

# k^-5/3 reference line anchored on the final spectrum
kref = np.array([3.0, float(kcut)])
Eref = 0.4 * spectrum_snaps[-1]["E"][min(3, kcut)]
axE.loglog(kref, Eref * (kref / 3.0) ** (-5 / 3), "k--", lw=1, label=r"$k^{-5/3}$")

axE.set_xlim(1, kcut)
axE.set_xlabel("wavenumber k")
axE.set_ylabel("E(k)")
axE.set_title("Energy spectrum")
axE.legend()
axE.grid(True, which="both", alpha=0.2)

axP.axhline(0, color="k", lw=0.5)
axP.set_xlim(0, kcut)
axP.set_xlabel("wavenumber k")
axP.set_ylabel("Pi(k)")
axP.set_title("Energy flux through k (>0: large to small scales)")
axP.legend()
axP.grid(alpha=0.2)

if SHOW_ANIMATION and "w" in spectrum_snaps[0]:
    from matplotlib import animation
    from IPython.display import HTML, display

    vmax = np.percentile(np.concatenate([s["w"].ravel() for s in spectrum_snaps]), 99.5)
    fig_a, (axw, axE2, axP2) = plt.subplots(1, 3, figsize=(16, 4.8), layout="constrained")
    im = axw.imshow(spectrum_snaps[0]["w"].T, origin="lower", cmap="inferno",
                    interpolation="bilinear", vmin=0, vmax=vmax, extent=[0, 2 * np.pi, 0, 2 * np.pi])

    def draw(i):
        s = spectrum_snaps[i]
        im.set_data(s["w"].T)
        axw.set_xlabel(f"t = {s['t']:.2f}")
        axE2.clear()
        axE2.loglog(kk[1:], s["E"][1:], "C0", lw=2.5)
        axE2.loglog(kref, Eref * (kref / 3.0) ** (-5 / 3), "k--", lw=1)
        axE2.set_xlim(1, kcut); axE2.set_yscale("log")
        axP2.clear()
        axP2.plot(kk, s["Pi"], "C3", lw=2.5)
        axP2.axhline(0, color="k", lw=0.5)
        axP2.set_xlim(0, kcut)
        return []

    anim = animation.FuncAnimation(fig_a, draw, frames=len(spectrum_snaps), interval=120, blit=False)
    display(HTML(anim.to_jshtml()))
    plt.close(fig_a)


## 4. Using your own saved data

If you already have snapshots on disk (for example from a longer run, or from `SAVE_NPZ = True` above), load them instead of re-running the DNS:

```python
dns = np.load("taylor_green_dns.npz", allow_pickle=True)

results, fig, axes = plot_turbulence_budget(
    velocity=dns["velocity"],                 # (nt, 3, nx, ny, nz)
    nu=float(dns["nu"]),
    times=dns["times"],
    domain_lengths=tuple(dns["domain_lengths"]),
    component_axis=1,
)
```

Notes:

- `*.npz` is listed in `.gitignore`: the `full` preset produces a **475 MB float32 array**, which compresses to a ~113 MB file. That is within Git's abilities but past GitHub's 100 MB per-file hard limit, and it would slow every clone forever. Regenerate it, or share a specific run as a [GitHub Release](https://docs.github.com/en/repositories/releasing-projects-on-github) asset.
- `allow_pickle=True` is only needed if the file stores an object array (older versions of this notebook saved a metadata dict that way). Prefer saving plain arrays so it can stay `False`.
- If your array stores the components last, `(nt, nx, ny, nz, 3)`, pass `component_axis=-1`.

Run the cell below to confirm the `results` keys you can rely on:


In [ ]:
for key, value in results.items():
    if key == "figures":
        print(f"{key:26s} -> dict with {sorted(value.keys())}")
    else:
        print(f"{key:26s} -> shape {value.shape}, finite: {bool(np.all(np.isfinite(value)))}")


## 5. Assumptions, limitations and things to check

**Grid and boundary conditions.** All derivatives are Fourier-based, so the analysis is only valid on a uniform Cartesian grid that is periodic in all three directions. It will silently produce wrong numbers on a non-periodic or stretched mesh.

**Production is zero by construction, not by measurement.** The budget is decomposed about the domain mean. The Taylor-Green initial condition has zero mean velocity, and a triply periodic domain without body forces keeps the domain-averaged momentum constant, so there is no mean shear to produce turbulence. For channel or boundary-layer data you must compute `P = -<u_i><u_j> dU_i/dx_j` from the mean profile instead; do not read this panel as "production is negligible in this flow".

**Transport.** `transport_mean` averages a divergent flux over a periodic domain and should come out at machine precision (~`1e-19` in the `full` run). Its RMS is the physically meaningful magnitude, which is why the panel plots both.

**Dealiasing.** With `dealias=True` the 2/3 rule is applied to the velocity *and* to every nonlinear product, so the reported dissipation is the dissipation of the resolved, filtered field. Comparing runs of different resolution needs care near the cutoff.

**Closure error.** `energy_budget_residual` measures integration and filtering error: it was ~`7e-6` m^2/s^2 (relative ~`6e-5`) on the `full` preset. Treat a growing residual as a sign that `dt` is too large or that snapshots are too far apart for the trapezoidal integration of `epsilon`.

**Memory.** `plot_turbulence_budget` converts its input to float64. A `full`-preset array is 475 MB as stored and ~950 MB after conversion, before the temporary gradient arrays are allocated. Slice the snapshot axis or process in chunks for larger datasets.

**Units.** The simulation is effectively in nondimensional (spectral code) units where lengths are in domain units and `nu = 1/400`. The axis labels carry m^2/s^2 and m^2/s^3 for readability; if you compare against the non-dimensional literature, drop the unit labels.

## 6. Figures

With `SAVE_FIG_PREFIX = "figures/demo_"` in section 2 the module writes `budget.png`, `closure.png` and `terms.png`. The PNGs committed in `figures/` were produced with `PRESET = "full"` at `N = 64`, so they are reproducible from this notebook alone.
